# Create Fed cuts and hikes dates history
Discount Rate Changes (DISCOUNT) from 02-02-1934 to 06-11-2002 (rate charged by the Fed to depositary institutions)
Federal Funds Target Rate (DFEDTAR) from 27-09-1982 to 15-12-2008 (target rate for overnight lending between banks)
Federal Funds Target Range Lower Limit (DFEDTARL) or Upper Limit (DFEDTARU) symmetric, both boundaries shift upward or downward with hikes and cuts respectively. The target range exists since 16 Dec 2008 as a consequence of the GFC.

For consistency, the FFR target is considered as of 27-09-1982. Please be aware...
- [Discount Rate Changes :](https://fred.stlouisfed.org/series/DISCOUNT) "Data before 1975 represent the date of the **New York Fed** discount rate change. After 1975, this corresponds to the Federal Reserve discount rate change".
- [Federal Funds Target Rate :](https://fred.stlouisfed.org/series/DFEDTAR) "Data for the period prior to 1994 come from the working paper 'A New Federal Funds Rate Target Series: September 27, 1982 - December 31, 1993' (Thornton, Federal Reserve Bank of St. Louis, 2005). Due to an error in the paper values from April 2, 1986 - April 20, 1986 were adjusted manually to 7.3125%. Data from 1994 to the present are derived from FOMC meeting transcripts and FOMC meeting statements."


In [ ]:
# Librairies
import json
import pandas as pd
from fismi import sdxData as sdx
import numpy as np
import requests
import io
from fismi import sdxData

In [12]:
# Hikes versus cuts 
def hikeOrCut(df):
    df = df.copy()
    df["Delta"] = df.iloc[:, 0].diff()
    df["Hike"] = df.index.where(df["Delta"] > 0)
    df["Cut"]  = df.index.where(df["Delta"] < 0)
    return df

In [14]:
# Get FOMC dates from Alfred (weird)
dls = "https://alfred.stlouisfed.org/release/downloaddates?rid=101&ff=xlsx"
filename = "fomcDates2000-Now.xlsx"
resp = requests.get(dls)
output = open(filename, 'wb')
output.write(resp.content)
output.close()

# Get names
xls = pd.ExcelFile(filename)
sheetName = xls.sheet_names[1]

# Dataframe...
fomcDates = pd.read_excel(filename, sheetName)
fomcDates["FOMC Meeting"] = fomcDates[sheetName]
fomcDates = fomcDates.set_index(sheetName, drop=True)
fomcDates

,FOMC Meeting
Release Dates,
2000-04-07,2000-04-07
2008-12-15,2008-12-15
2014-04-03,2014-04-03
2014-04-04,2014-04-04
2014-04-07,2014-04-07
...,...
2026-10-02,2026-10-02
2026-10-03,2026-10-03
2026-10-04,2026-10-04


In [15]:
# Query data ; API key
with open('/Users/lea_gosselin/.openbb_platform/user_settings.json', 'r') as file:
    keys = json.load(file)
    
key=keys["credentials"].get("fred_api_key") 

In [ ]:
# 4 series to collect : 
tickers = ["DISCOUNT", "DFEDTAR", "DFEDTARL", "DFEDTARU"]

results = {}

for tick in tickers:
    # Query
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={tick}&api_key={key}&file_type=json"
    df = sdxData.queryFredData(url)
    
    # Get hike and cut dates, compute policy spread
    df = hikeOrCut(df)
    
    # Set daily dates
    df = df.asfreq("D")
    
    # Store
    results[tick] = df

In [17]:
# Compute mid-point range after 16/12/2008
dfMid = results["DFEDTARU"]
dfMid["value"] = (results["DFEDTARU"]["value"]-results["DFEDTARL"]["value"])/2

In [18]:
# Merge results
sDate1 = results["DFEDTAR"].index[0]
sDate2 = dfMid.index[0]
dFinal = pd.concat([results["DISCOUNT"][results["DISCOUNT"].index < sDate1], 
                    results["DFEDTAR"][results["DFEDTAR"].index < sDate2],
                    dfMid])
dFinal

,value,Delta,Hike,Cut
date,,,,
1934-02-02,1.500,NaN,NaT,NaT
1934-02-03,NaN,NaN,NaT,NaT
1934-02-04,NaN,NaN,NaT,NaT
1934-02-05,NaN,NaN,NaT,NaT
1934-02-06,NaN,NaN,NaT,NaT
...,...,...,...,...
2026-10-02,0.125,0.0,NaT,NaT
2026-10-03,0.125,0.0,NaT,NaT
2026-10-04,0.125,0.0,NaT,NaT


In [19]:
# # Add FOMC dates (flag even if nothing happened)
# dFinal = dFinal.merge(
#     fomcDates,
#     how="left",
#     right_index=True,
#     left_index=True
# )
# dFinal

In [37]:
# Statistiques
dFinal["Month"] = dFinal.index.to_period("Y").to_timestamp()
A = dFinal.groupby("Month")[["Hike", "Cut"]].count()
A["Total"] = A.sum(axis=1)
A

,Hike,Cut,Total
Month,,,
1934-01-01,0,0,0
1935-01-01,0,0,0
1936-01-01,0,0,0
1937-01-01,0,1,1
1938-01-01,0,0,0
...,...,...,...
2022-01-01,7,0,7
2023-01-01,4,0,4
2024-01-01,0,3,3


In [ ]:
dFinal["Month"] = dFinal["Month"].dt.month()

In [20]:
# Save...
dFinal.to_excel("fomcDatesHistory.xlsx")